# Data manipulation

<a id='Index'></a>
# Exercise list:
* <a href='#Exercise-01'>Exercise-01</a>
* <a href='#Exercise-02'>Exercise-02</a>
* <a href='#Exercise-03'>Exercise-03</a>
* <a href='#Exercise-04'>Exercise-04</a>
* <a href='#Exercise-05'>Exercise-05</a>
* <a href='#Exercise-06'>Exercise-06</a>
* <a href='#Exercise-07'>Exercise-07</a>
* <a href='#Exercise-08'>Exercise-08</a>
* <a href='#Exercise-09'>Exercise-09</a>
* <a href='#Exercise-10'>Exercise-10</a>
* <a href='#Exercise-11'>Exercise-11</a>

## Main goals:
* Merge weather and train data with Pandas and Spark
* Create a "training" and a "test" dataset for ML algorithms later on

* We start a Spark session overwriting the default values:
  * --total-executor-cores 36 
  * --executor-memory 1g  
* Other default values:
   * --name "Jupyter PySpark"
   * ...

In [1]:
%spark 16 32g

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/01 15:24:54 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.4.2
      /_/

Using Python version 3.9.18 (main, Dec 15 2023 17:48:57)
Spark context Web UI available at None
Spark context available as 'sc' (master = local[*], app id = local-1759325095354).
SparkSession available as 'spark'.


* Run the Handlers to initialise classes

In [2]:
%run handlers.ipynb

Pandas version:  2.1.4


In [3]:
# Print a file with all requirements
!pip freeze > requirements.txt

# Start manipulation

* Start reading in **train** and **weather** files with **Spark**

* Objective: Create the DataFrames df_sched and df_act (Spark) based on the read-in data.

In [4]:
filename_sched = '20170901-20171019_scheduled_S-Bahn_Stuttgart.csv'
path_sched = os.path.join(basedir, filename_sched)

filename_act = '20170901-20171019_actual_S-Bahn_Stuttgart.csv'
path_act = os.path.join(basedir, filename_act)

filename_weather = 'S-Mitte-SZ-30min-values_2017.xls'
path_weather = os.path.join(basedir, filename_weather)

# Read in train files (csv) with Spark, and drop some columns:
df_sched = spark.read.option("header", True).option("sep", ";").csv(os.path.join(LustrePrefix,path_sched))
df_sched = df_sched.na.drop(how = 'any')

df_act = spark.read.option("header", True).option("sep", ";").csv(os.path.join(LustrePrefix,path_act))
df_act = df_act.na.drop(how = 'any')

df_sched = df_sched.drop('ZUGEREIGNIS_VERW_ID', 'ZUGEREIGNIS_BETR_KUERZEL', 'UGEREIGNIS_BETR_KUERZEL', \
                       'ZUGEREIGNIS_BETREIBER', 'ZUGEREIGNIS_ZUGGATTUNG', 'ZUGEREIGNIS_RELEVANT',\
                       'ZUGEREIGNIS_BETR_KURZ', 'ZUGEREIGNIS_GLEIS_SOLL')

df_act = df_act.drop('ZUGEREIGNIS_ZUGGATTUNG', 'QUELLE_SENDER')

## Useful reading:

To **visualise size and structure** of the generated DataFrames:

**Spark** (e.g. dataframe df_sched):
* print(df_sched.count()): Number of rows in the DataFrame
* print(len(df_sched.columns)): Length of the list containing the names of the columns
* df_sched.show(n=5)
* **!!!** Without an optimised Spark framework these operations might lead to a kernel crash **!!!**

**Pandas** (e.g. dataframe df_pd_weather below):
* print(df_pd_weather.shape): Tuple representing the dimensions of the DataFrame (rows, columns)
* df_pd_weather.iloc[:5,:] : Slice the DataFrame by **implicit** indexing of **[rows, columns]**
* df_pd_weather.loc[:5,:] : Slice by **explicit** indexing, i.e., the given names of rows and columns


* In these exercises, Pandas DataFrames will have suffix **\_pd**

<a href='#Index'>Exercise Index</a>
<a id='Exercise-01'></a>

# EXERCISE 1
* Try out the previous commands with df_sched (Spark DF) **after conversion to Pandas**
* Display e.g. 10 rows and all the columns of df_sched_pd
* **Separate the next block into two blocks** with Ctrl-ALt-Minus (could be useful afterwards)

In [5]:
df_sched_pd = df_sched.toPandas()
df_sched_pd.iloc[0:10, :]

,SERVICE_ID,SERVICE_START_ZEIT,SERVICE_HALTNR,ZUGEREIGNIS_ZUGNUMMER,ZUGEREIGNIS_DS100,ZUGEREIGNIS_TYP,ZUGEREIGNIS_SOLLZEIT,ZUGEREIGNIS_EVANR,ZUGEREIGNIS_LINIE
0,30064857037,31.08.2017 23:21:00,26,7177,TSC,20,01.09.2017,8005769,1
1,30120708901,01.09.2017,1,7178,TBO,10,01.09.2017,8001055,1
2,30471704613,31.08.2017 23:04:00,44,7272,TGC,20,01.09.2017,8002448,2
3,30471704613,31.08.2017 23:04:00,45,7272,TGC,40,01.09.2017,8002448,2
4,30480093223,31.08.2017 23:34:00,18,7274,TSS,20,01.09.2017,8006698,2
5,30480093223,31.08.2017 23:34:00,19,7274,TSS,40,01.09.2017,8006698,2
6,30484287530,31.08.2017 23:18:00,34,7277,TSS,20,01.09.2017,8006698,2
7,30484287530,31.08.2017 23:18:00,35,7277,TSS,40,01.09.2017,8006698,2
8,30492676140,31.08.2017 23:48:00,10,7279,TBTB,20,01.09.2017,8000934,2
9,30492676140,31.08.2017 23:48:00,11,7279,TBTB,40,01.09.2017,8000934,2


**Reference page for Spark**
* https://spark.apache.org/docs/latest/api/python/
* Use "Search the docs" on the left.
* The needed solution usually follows: "pyspark.sql."

**Reference page for Pandas**
* https://pandas.pydata.org/pandas-docs/stable/index.html
* Use "Search the docs" up on the left.
* Needed solution usually follows: "pandas.DataFrame"
* Look in the Parameters list for the parameters we need.

# Create a dictionary of station names with Pandas

## EXERCISE 1 (cont'd)
* Use iloc\[\__,__\]  to display, e.g., **10 rows** and **all columns** of the DataFrame "bahnhofs"

In [6]:
# Read in data of stations "StationData.csv" as a pandas DataFrame
filename_bahnhof_csv = 'StationData.csv'
path_bahnhof_csv = os.path.join(basedir, filename_bahnhof_csv)
bahnhofs = pd.read_csv(os.path.join(LustrePrefix,path_bahnhof_csv), sep = ';', encoding = "ISO-8859-1")

#Use iloc:
bahnhofs.iloc[0:10, :]

,ID,DS100,NAME
0,8000002,TA,Aalen Hbf
1,8000014,TAU,Aulendorf
2,8000016,TB,Backnang
3,8000017,TBF,Bad Friedrichshall Hbf
4,8000038,TBM,Bietigheim-Bissingen
5,8000063,TCW,Calw
6,8000067,TC,Crailsheim
7,8000069,TSIG,Sigmaringen
8,8000096,TS,Stuttgart Hbf
9,8000101,TET,Eutingen im Gäu


* The "index" attribute gives access to the **index labels of the rows**
* Indices are the very first column of the DataFrame
* Use index\[\__\] to display the n-th index (starting from 0) of the DataFrame, or a range of indices
* (If you try,) you will notice you cannot directly modify an index

In [7]:
#print(bahnhofs.index[0])
#bahnhofs.index[0]= __

* Use columns\[\__\] to display all or a subset of the **column labels** of the DataFrame

In [8]:
print(bahnhofs.columns)

Index(['ID', 'DS100', 'NAME'], dtype='object')


**INDEX change**
* Use **set_index** to set the 'DS100'-codes as row index of the DataFrame (instead of the numbers 0,1,2,...)

In [9]:
bahnhofs = bahnhofs.set_index('DS100')
print(bahnhofs.index[:])

Index(['TA', 'TAU', 'TB', 'TBF', 'TBM', 'TCW', 'TC', 'TSIG', 'TS', 'TET',
       ...
       'TDH', 'TUMS', 'TKAUB', 'TMBSK', 'TWEB', 'TMW', 'TNUV', 'TNU R',
       'TFKHK', 'TS  T'],
      dtype='object', name='DS100', length=579)


* From the DataFrame **bahnhofs**, slice only the **'NAME'** column of the DataFrame

In [10]:
abbr_bahnhofs=bahnhofs['NAME']
print(abbr_bahnhofs[0:10])

# We obtain a Series with indices DS100 and values NAME

DS100
TA                   Aalen Hbf
TAU                  Aulendorf
TB                    Backnang
TBF     Bad Friedrichshall Hbf
TBM       Bietigheim-Bissingen
TCW                       Calw
TC                  Crailsheim
TSIG               Sigmaringen
TS               Stuttgart Hbf
TET            Eutingen im Gäu
Name: NAME, dtype: object


**CONVERT TO DICTIONARY**
* Use **to_dict()** to convert the Pandas Series into a Python dictionary of abbreviations

In [11]:
abbr_bahnhofs_dic=abbr_bahnhofs.to_dict()

print('Some dictionary key (DS100) + value (full name):')
for i in list(range(5)):
    print(list(abbr_bahnhofs_dic.items())[i][0], ': ', list(abbr_bahnhofs_dic.items())[i][1])
print('...')

Some dictionary key (DS100) + value (full name):
TA :  Aalen Hbf
TAU :  Aulendorf
TB :  Backnang
TBF :  Bad Friedrichshall Hbf
TBM :  Bietigheim-Bissingen
...


# END OF EXERCISE 1

# Manipulation of weather data with Pandas 
* Read in the September and October weather files (Excel tables) with **Pandas**. 

* Objective: Obtain separate Pandas DFs for both September and October (df_sep_pd_weather, df_oct_pd_weather).

In [12]:
# Read-in weather files
df_pd_weather = pd.ExcelFile(os.path.join(LustrePrefix,path_weather))
#df_pd_weather is a DataFrame: create 2 DataFrames corresponding to different months.
df_sep_pd_weather = pd.read_excel(df_pd_weather, 'Sept. 2017')
df_oct_pd_weather = pd.read_excel(df_pd_weather, 'Okt. 2017')

* Create lists of column names: Needed columns as well as those to be dropped.

In [13]:
column_name = ['Datum','Uhrzeit', 'Mittelwerte_Temperatur', 'Maxwerte_Temperatur', 'Minwerte_Temperatur',\
               'Mittelwerte_Rel_Feuchte', 'Mittel_WG', 'Max_WG', 'Mittel_WR', 'Mittel_Druck', 'Summe_Niederschalg',\
               'Mittel_Globalstr', 'Mittel_Str_Bilanz', 'Mittel_UVA_Str', 'Mittel_UVB_Str', 'Mittel_NO', 'Mittel_NO2',\
               'Mittel_O3', 'Mittel_PM10', 'Mittel_PM2.5']
# The column "Datum" contains also the time (although maybe not directly visible from the Excel cells). 
# Therefore we can drop "Uhrzeit", as well as other data which are not needed.
column_drop = ['Uhrzeit','Maxwerte_Temperatur', 'Minwerte_Temperatur', 'Mittel_WR', 'Mittel_Globalstr', 'Mittel_Str_Bilanz',\
               'Mittel_UVA_Str', 'Mittel_UVB_Str', 'Mittel_Druck','Mittel_PM2.5']

* More weather elaboration with **Pandas**: Concatenate data of September and October in a unique DataFrame.
* Objective: df_pd_weather (**Pandas**).

In [14]:
# Define an index for the columns:
df_sep_pd_weather.columns = column_name
df_oct_pd_weather.columns = column_name

# Concatenate the two sets adding by the rows. 
# A new integer index for the DataFrame rows is created (0,1,...)
df_pd_weather = pd.concat([df_sep_pd_weather, df_oct_pd_weather], ignore_index = True)

* More weather elaboration with **Pandas** to delete non-valid values, e.g. not-a-numbers (NaNs).
* Objective: df_pd_weather (**Pandas**).

In [15]:
# Access data via name of column: e.g. ".Uhrzeit" or "['Mittelwerte_T']" 
# and delete rows corresponding to empty fields (either notnull() or notna() can be used).
# There could still be NaNs in other columns which will be deleted at a later stage.
df_pd_weather = df_pd_weather[df_pd_weather.Uhrzeit.notnull()]
df_pd_weather = df_pd_weather[df_pd_weather['Mittelwerte_Temperatur'].notnull()]

print(df_pd_weather.shape)
df_pd_weather.iloc[:5,:]

(2932, 20)


,Datum,Uhrzeit,Mittelwerte_Temperatur,Maxwerte_Temperatur,Minwerte_Temperatur,Mittelwerte_Rel_Feuchte,Mittel_WG,Max_WG,Mittel_WR,Mittel_Druck,Summe_Niederschalg,Mittel_Globalstr,Mittel_Str_Bilanz,Mittel_UVA_Str,Mittel_UVB_Str,Mittel_NO,Mittel_NO2,Mittel_O3,Mittel_PM10,Mittel_PM2.5
5,Datum,Uhrzeit,Temperatur (°C),Temperatur (°C),Temperatur (°C),Rel. Feuchte (%),WG (m/s),WG (m/s),WR (Grad),Druck (hPa),Niederschlag (l/m²),Globalstr. (W/m²),Str.-Bilanz (W/m²),UVA-Str. (W/m²),UVB-Str. (W/m²),NO (µg/m³),NO2 (µg/m³),O3 (µg/m³),PM10 (µg/m³),"PM2,5 (µg/m³)"
6,2017-09-01 00:30:00,00:30:00,15.1,15.3,15,85.5,0.8,1.5,245.2,989.4,0.2,0,-41.5,1.07,0.025,0.6,15.1,25.6,0,NaN
7,2017-09-01 01:00:00,01:00:00,14.9,15.1,14.7,87.1,0.7,1.5,244.1,989.5,0.5,0,-36.1,1.08,0.025,0.9,13.6,24.4,0,NaN
8,2017-09-01 01:30:00,01:30:00,14.8,15,14.6,87.4,0.8,1.4,283.3,989.4,0.8,0,-36.6,1.09,0.025,0,8.4,30.4,0,NaN
9,2017-09-01 02:00:00,02:00:00,14.7,14.9,14.5,87.5,0.4,0.7,311.9,989.1,0.2,0,-35.3,1.1,0.025,0.7,12,26.4,0,NaN


<a href='#Index'>Exercise Index</a>
<a id='Exercise-02'></a>

**EXERCISE 2**

In [16]:
# Look closely into the DataFrame above. What could be the issue there? (first row of data...)

# Empty space before the answer 


































# Answer:
# The headings are repeated in the first row of data!
# Delete the row corresponding to the first row of data by 
# - slicing AS IN THE PREVIOUS CELL
# - and using the conditional statement
df_pd_weather = df_pd_weather[df_pd_weather.Datum != 'Datum']
# We want to get rid of the row where the Datum is not...?

In [17]:
# Check your result with:
print(df_pd_weather.shape)
df_pd_weather.iloc[:5,:]

(2930, 20)


,Datum,Uhrzeit,Mittelwerte_Temperatur,Maxwerte_Temperatur,Minwerte_Temperatur,Mittelwerte_Rel_Feuchte,Mittel_WG,Max_WG,Mittel_WR,Mittel_Druck,Summe_Niederschalg,Mittel_Globalstr,Mittel_Str_Bilanz,Mittel_UVA_Str,Mittel_UVB_Str,Mittel_NO,Mittel_NO2,Mittel_O3,Mittel_PM10,Mittel_PM2.5
6,2017-09-01 00:30:00,00:30:00,15.1,15.3,15,85.5,0.8,1.5,245.2,989.4,0.2,0,-41.5,1.07,0.025,0.6,15.1,25.6,0,NaN
7,2017-09-01 01:00:00,01:00:00,14.9,15.1,14.7,87.1,0.7,1.5,244.1,989.5,0.5,0,-36.1,1.08,0.025,0.9,13.6,24.4,0,NaN
8,2017-09-01 01:30:00,01:30:00,14.8,15,14.6,87.4,0.8,1.4,283.3,989.4,0.8,0,-36.6,1.09,0.025,0,8.4,30.4,0,NaN
9,2017-09-01 02:00:00,02:00:00,14.7,14.9,14.5,87.5,0.4,0.7,311.9,989.1,0.2,0,-35.3,1.1,0.025,0.7,12,26.4,0,NaN
10,2017-09-01 02:30:00,02:30:00,14.7,15,14.5,87.2,0.6,1,307.3,988.8,0.1,0,-42,1.1,0.025,1.6,13,25,0,NaN


* Drop the columns that are not needed:

In [18]:
df_pd_weather = df_pd_weather.drop(column_drop, axis = 1)

<a href='#Index'>Exercise Index</a>
<a id='Exercise-03'></a>

**EXERCISE 3**
* Adjust the weather DataFrame by filling in missing values with the mean value of the column (this might not always be the optimal procedure!):

In [19]:
# Relevant columns:
# WG (WindGeschwindigkeit) = Wind velocity
# Rel_Feuchte = Humidity
# Summe_Niederschalg = Total precipitation
# ... (see lecture slides "Source Data" for further translations...)
names_ = ['Mittelwerte_Temperatur', 'Mittelwerte_Rel_Feuchte', 'Mittel_WG','Max_WG', \
          'Summe_Niederschalg', 'Mittel_NO', 'Mittel_NO2', 'Mittel_O3', 'Mittel_PM10']

for name in names_:
#   Convert data to float64 (or int64)
    df_pd_weather[name] = pd.to_numeric(df_pd_weather[name],errors = 'coerce') #coerce = invalid parsing will be set as NaN
#   Compute the mean value of the column:
    mean_weather = df_pd_weather[name].mean()

#    Replace with the mean value of the column in case of NaN (use the function fillna):
    df_pd_weather[name] = df_pd_weather[name].fillna(value=mean_weather)

In [20]:
# Check your result with:
print(df_pd_weather.shape)
df_pd_weather.iloc[:5,:]

(2930, 10)


,Datum,Mittelwerte_Temperatur,Mittelwerte_Rel_Feuchte,Mittel_WG,Max_WG,Summe_Niederschalg,Mittel_NO,Mittel_NO2,Mittel_O3,Mittel_PM10
6,2017-09-01 00:30:00,15.1,85.5,0.8,1.5,0.2,0.6,15.1,25.6,0.0
7,2017-09-01 01:00:00,14.9,87.1,0.7,1.5,0.5,0.9,13.6,24.4,0.0
8,2017-09-01 01:30:00,14.8,87.4,0.8,1.4,0.8,0.0,8.4,30.4,0.0
9,2017-09-01 02:00:00,14.7,87.5,0.4,0.7,0.2,0.7,12.0,26.4,0.0
10,2017-09-01 02:30:00,14.7,87.2,0.6,1.0,0.1,1.6,13.0,25.0,0.0


# Manipulation of weather data with Spark 
* We manipulated and wrote from Pandas, now we read-in the same weather data as **Spark** files.
* The following is one way to convert Pandas into Spark DataFrames (pandas -> csv -> spark: probably not the most efficient one)
* Objective: df_weather (Spark) for further manipulation.

* Save the **Pandas** weather DataFrame locally as a csv file.

In [21]:
filename = 'weather_csv.csv'
path = os.path.join(local_data_dir, filename)

print("Writing down df_pd_weather in...")
start_time=time.time()

# Write object to a comma-separated-values (csv) file:
# Use to_csv to write down the DataFrame we just generated (with default encoding).
# Look up to_csv in: https://pandas.pydata.org/pandas-docs/stable/index.html

df_pd_weather.to_csv(path, encoding = 'utf-8')
print("--- %.0f min. %.2f sec.---" % (np.floor_divide(time.time() - start_time, 60),
                                          np.remainder(time.time() - start_time, 60)))

Writing down df_pd_weather in...
--- 0 min. 0.03 sec.---


* You can browse the Notebooks homepage to check if the file is now present in the folder NB_Dataframes.

In [22]:
# Read-in the weather data as a Spark file:

# Generate a Spark DataFrame:
df_weather = spark.read.option("header", True).csv(os.path.join(LustrePrefix,path))
# header=TRUE : use the first row as names of columns

# A column of indices called "_c0" is created in the conversion. It replaces the (implicit) row indices in pandas.
# This column can be dropped.
df_weather = df_weather.drop('_c0')

<a href='#Index'>Exercise Index</a>
<a id='Exercise-04'></a>

**EXERCISE 4**
* More weather manipulation with Spark.

In [23]:
# Drop columns in column_drop (already done, redundant):
df_weather = df_weather.select([c for c in df_weather.columns if c not in column_drop])

# "col()" is a spark.sql function: Returns a column based on the given column name.
# Use "where", "col", and "isNotNull" to drop all rows of df_weather that contains an empty DATUM.
df_weather = df_weather.where(col('DATUM').isNotNull())

In [24]:
# Check your result with:
print((df_weather.count(), len(df_weather.columns)))
df_weather.show(n=5)
# We see that visualisation with Spark is not as nice as with Pandas!

(2928, 10)
+-------------------+----------------------+-----------------------+---------+------+------------------+---------+----------+---------+-----------+
|              Datum|Mittelwerte_Temperatur|Mittelwerte_Rel_Feuchte|Mittel_WG|Max_WG|Summe_Niederschalg|Mittel_NO|Mittel_NO2|Mittel_O3|Mittel_PM10|
+-------------------+----------------------+-----------------------+---------+------+------------------+---------+----------+---------+-----------+
|2017-09-01 00:30:00|                  15.1|                   85.5|      0.8|   1.5|               0.2|      0.6|      15.1|     25.6|        0.0|
|2017-09-01 01:00:00|                  14.9|                   87.1|      0.7|   1.5|               0.5|      0.9|      13.6|     24.4|        0.0|
|2017-09-01 01:30:00|                  14.8|                   87.4|      0.8|   1.4|               0.8|      0.0|       8.4|     30.4|        0.0|
|2017-09-01 02:00:00|                  14.7|                   87.5|      0.4|   0.7|               0

# Manipulation of S-Bahn data with Spark 

<a href='#Index'>Exercise Index</a>
<a id='Exercise-05'></a>

**EXERCISE 5**
* Join the Spark DataFrames df_act (**real data**) and df_sched (**scheduled data**) into one DataFrame df_all.

In [25]:
# Merge the two DataFrames where data correspond at the columns in the list below (must be present in BOTH DataFrames).
# (See also lecture slides "Source Data" with the list of feature columns for translation and explanations)

columns = ['ZUGEREIGNIS_SOLLZEIT','SERVICE_ID', 'ZUGEREIGNIS_TYP',\
           'ZUGEREIGNIS_ZUGNUMMER', 'ZUGEREIGNIS_DS100']

# Use "join()" to merge df_act and df_sched at corresponding journeys.
# For more information on this function, consult https://spark.apache.org/docs/latest/api/python
# The resulting DataFrame is df_all:
df_all = df_act.join(df_sched, columns, "inner")

<a href='#Index'>Exercise Index</a>
<a id='Exercise-06'></a>

**EXERCISE 6**

* In df_all, get rid of all rows with at least one missing value.

In [26]:
# In https://spark.apache.org/docs/latest/api/python/ :
# Look for DataFrameNaFunctions and how to use the function "drop".  

# "How" option in drop:
# If ‘any’: drop a row if it contains any nulls.
# If ‘all’: drop a row only if all its values are null.

# Now, get rid of rows with at least one missing value in df_all:
df_all = df_all.na.drop(how='any')

* Select only some of the columns, and remove duplicated rows:

In [27]:
# We want to generate a new DataFrame containing selected columns and unique rows of df_all.
# We concatenate:
# - "select()" to retain only the columns in the argument list AND
# - "distinct()" to delete identical rows
columns = ['ZUGEREIGNIS_LINIE','SERVICE_START_ZEIT', 'ZUGEREIGNIS_ISTZEIT','ZUGEREIGNIS_TYP',\
           'ZUGEREIGNIS_SOLLZEIT','SERVICE_ID', 'ZUGEREIGNIS_ZUGNUMMER', 'ZUGEREIGNIS_DS100']

df_all = df_all.select(columns).distinct()

In [28]:
# Commented out since Spark IO might cause kernel crash.
# Check your result with (maybe after the course):
print((df_all.count(), len(df_all.columns)))
df_all.show(n=5)

(1091426, 8)


+-----------------+-------------------+-------------------+---------------+--------------------+-----------+---------------------+-----------------+
|ZUGEREIGNIS_LINIE| SERVICE_START_ZEIT|ZUGEREIGNIS_ISTZEIT|ZUGEREIGNIS_TYP|ZUGEREIGNIS_SOLLZEIT| SERVICE_ID|ZUGEREIGNIS_ZUGNUMMER|ZUGEREIGNIS_DS100|
+-----------------+-------------------+-------------------+---------------+--------------------+-----------+---------------------+-----------------+
|                1|31.08.2017 23:21:00|         01.09.2017|             20|          01.09.2017|30064857037|                 7177|              TSC|
|                2|31.08.2017 23:04:00|         01.09.2017|             20|          01.09.2017|30471704613|                 7272|              TGC|
|                2|31.08.2017 23:34:00|         01.09.2017|             20|          01.09.2017|30480093223|                 7274|              TSS|
|                4|31.08.2017 23:55:00|         01.09.2017|             20|          01.09.2017|3139025743

<a href='#Index'>Exercise Index</a>
<a id='Exercise-07'></a>

**EXERCISE 7**
* In the DataFrame, time is indicated as date (DD.MM.YYYY) and clock (HH:MM:SS), e.g. 01.09.2017 00:26:00
* This format clearly does not allow for operations on time!
* For this reason, time must be converted into **unix time**.
* Then, we can easily compute e.g. the **delay** and the **duration of service**. 
* Objective: df_all (Spark) with columns **VERSPAETUNG** (delay) and **LAUFSZEIT** (duration) in minutes.

In [29]:
# E.g. 01.09.2017 00:26:00
# corresponds to the Unix timestamp: 1504225560 (seconds from 01/01/1970 00:00)
# See https://www.unixtimestamp.com

# Below:
# - withColumn() returns a new DataFrame by adding or replacing the column with a new set of data 
# - udfConvertDateToUnix() is a user-defined function (in handlers) which converts a time-entry into a unix time stamp
# The time of a scheduled event (ZUGEREIGNIS_SOLLZEIT) is converted to unix-time: 
df_all = df_all\
    .withColumn("ZUGEREIGNIS_SOLLZEIT", udfConvertDateToUnix("ZUGEREIGNIS_SOLLZEIT"))

# Add in one line of code (with a linebreak) the conversion to unix-time for
# - time of an actual event  (column ZUGEREIGNIS_ISTZEIT)
# - time of start of service (column SERVICE_START_ZEIT)
df_all = df_all\
    .withColumn("ZUGEREIGNIS_ISTZEIT", udfConvertDateToUnix("ZUGEREIGNIS_ISTZEIT"))\
    .withColumn("SERVICE_START_ZEIT", udfConvertDateToUnix("SERVICE_START_ZEIT"))

In [30]:
# Commented out since Spark IO might cause kernel crash.
# Slice e.g. one interesting column and show:
df_all.select(['ZUGEREIGNIS_SOLLZEIT']).show(n=5)

+--------------------+
|ZUGEREIGNIS_SOLLZEIT|
+--------------------+
|          1504216800|
|          1504216800|
|          1504216800|
|          1504216800|
|          1504216920|
+--------------------+
only showing top 5 rows



* Compute **8 additional time features** derived from the unix time.
* The built-in function from_unixtime() converts the unix time into a time string.
* It can be combined with the built-in Spark functions for month, hour, minute, day of month.

In [31]:
# Originally one time string (e.g. 2019-10-19 13:15:30), then a unix stamp, then several features:

df_all = df_all.\
withColumn('MONAT', month(from_unixtime(col('ZUGEREIGNIS_SOLLZEIT')))).\
withColumn('TAG', dayofmonth(from_unixtime(col('ZUGEREIGNIS_SOLLZEIT')))).\
withColumn('STUNDE_SOLL', hour(from_unixtime(col('ZUGEREIGNIS_SOLLZEIT')))).\
withColumn('MINUTE_SOLL', minute(from_unixtime(col('ZUGEREIGNIS_SOLLZEIT')))).\
withColumn('STUNDE_IST', hour(from_unixtime(col('ZUGEREIGNIS_ISTZEIT')))).\
withColumn('MINUTE_IST', minute(from_unixtime(col('ZUGEREIGNIS_ISTZEIT')))).\
withColumn('STUNDE_SER', hour(from_unixtime(col('SERVICE_START_ZEIT')))).\
withColumn('MINUTE_SER', minute(from_unixtime(col('SERVICE_START_ZEIT'))))

* There is no direct built-in function to obtain the feature "weekday"
* We can use instead **date_format(..., 'E')**
  * See e.g. https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html

In [32]:
df_all = df_all.\
withColumn('WOCHENTAG_IST', date_format(from_unixtime(col('ZUGEREIGNIS_ISTZEIT')),'E')).\
withColumn('WOCHENTAG_SER', date_format(from_unixtime(col('SERVICE_START_ZEIT')),'E')).\
withColumn('WOCHENTAG_SOLL', date_format(from_unixtime(col('ZUGEREIGNIS_SOLLZEIT')), 'E'))

In [33]:
# Commented out since Spark IO might cause kernel crash:
# You can also check that the corresponding day in 2017 is correct:
df_all.select(['MONAT','TAG','WOCHENTAG_IST','WOCHENTAG_SER','WOCHENTAG_SOLL']).show(n=5)

+-----+---+-------------+-------------+--------------+
|MONAT|TAG|WOCHENTAG_IST|WOCHENTAG_SER|WOCHENTAG_SOLL|
+-----+---+-------------+-------------+--------------+
|    9|  1|          Fri|          Thu|           Fri|
|    9|  1|          Fri|          Thu|           Fri|
|    9|  1|          Fri|          Thu|           Fri|
|    9|  1|          Fri|          Thu|           Fri|
|    9|  1|          Fri|          Thu|           Fri|
+-----+---+-------------+-------------+--------------+
only showing top 5 rows



* Assign a **minute range** to **match the S-Bahn data with the weather data** (measured every 30 minutes):
* 0 if minute_sched<30, 30 if minute_sched>=30.
* The function udfMinuteRange is a user-defined function in the handlers.

In [34]:
df_all = df_all.withColumn('MINUTE_RANGE', udfMinuteRange('MINUTE_SOLL'))

<a href='#Index'>Exercise Index</a>
<a id='Exercise-08'></a>

**EXERCISE 8**

* Compute the delay (VERSPAETUNG) and the duration of the journey (LAUFSZEIT) in **MINUTES**:

In [35]:
# - VERSPAETUNG = Delay, difference between actual event time (ZUGEREIGNIS_ISTZEIT)
#   and scheduled event time (ZUGEREIGNIS_SOLLZEIT)
# - LAUFSZEIT = Duration, difference between scheduled event time (ZUGEREIGNIS_SOLLZEIT)
#   and scheduled service start (SERVICE_START_ZEIT)

# The corresponding columns are the unix time values in seconds.
# .cast('int') is needed at the end to obtain an integer number of minutes.

df_all = df_all.withColumn('VERSPAETUNG', ((df_all['ZUGEREIGNIS_ISTZEIT'] - df_all['ZUGEREIGNIS_SOLLZEIT']) / 60).cast('int'))
df_all = df_all.withColumn('LAUFSZEIT', ((df_all['ZUGEREIGNIS_SOLLZEIT'] - df_all['SERVICE_START_ZEIT']) / 60).cast('int'))

# Drop some columns that we will not further need:
df_all = df_all.drop('STUNDE_IST', 'MINUTE_IST', 'WOCHENTAG_IST')

In [36]:
# Commented out since Spark IO might cause kernel crash.
if (False):
    # Check the two new columns:
    df_all.select(['VERSPAETUNG','LAUFSZEIT']).show(n=10)
    # Can you see negative delays?

* Similar **time** operations are carried out on the **weather** data.

* Objective: df_weather (Spark) with new time columns.

In [37]:
df_weather = df_weather.withColumn('TimeUnix', udfConvertDateToUnix(col('Datum'))) #or: udfConvertDateToUnix('Datum'))

df_weather = df_weather.\
withColumn('MONAT',month(from_unixtime(col('TimeUnix')))).\
withColumn('TAG', dayofmonth(from_unixtime(col('TimeUnix')))).\
withColumn('STUNDE_SOLL', hour(from_unixtime(col('TimeUnix')))).\
withColumn('MINUTE_RANGE', minute(from_unixtime(col('TimeUnix'))))

# Merge S-Bahn and weather data
* Produce a joint S-Bahn+weather Spark DataFrame (**df_proper**):

In [38]:
# df_all = S-Bahn "actual" + "scheduled" data (created above)

# df_proper: 
# Train and weather data are merged when data in the selected columns match,
# i.e., when both a weather measurement and a train event happen in the same time range
# (columns must exist in BOTH dataframes).
df_proper = df_all.join(df_weather, ['MONAT','TAG','STUNDE_SOLL','MINUTE_RANGE'])

In [39]:
# Commented out since Spark IO might cause kernel crash.
if (False):
    # Show that df_proper contains now both train and weather (matching) information
    df_proper.select(['MONAT','TAG','SERVICE_ID','VERSPAETUNG','LAUFSZEIT','Mittelwerte_Temperatur']).show(n=5)

In [40]:
# Commented out since Spark IO might cause kernel crash.
if (False):
    # Check size of df_proper BEFORE filtering:
    print((df_proper.count(), len(df_proper.columns)))

<a href='#Index'>Exercise Index</a>
<a id='Exercise-09'></a>

**EXERCISE 9**
* Filter out outstanding delays and delays "in advance".
* The function **filter()** **keeps** the row if the condition is satisfied.

In [41]:
# The column VERSPAETUNG contains the delay values in MINUTES:
# Use "filter()" to eliminate in one line:
# - trains IN ADVANCE (= negative delay)
# - delays higher than 3 hours 

# !! Condition satisfied => entries are kept !!
df_proper = df_proper.filter(col('VERSPAETUNG') >= 0).filter(col('VERSPAETUNG') <= 180)

* Keep also events only of **type "departure from a station"**:

In [42]:
# We are only interested in the departure time from each station (i.e. ZUGEREIGNIS_TYP = 40):
df_proper = df_proper.filter(col('ZUGEREIGNIS_TYP') == 40 )

# Remove duplicate rows again (if any):
df_proper = df_proper.distinct()

In [43]:
# Commented out since Spark IO might cause kernel crash.
if (False):
    # Check size of df_proper AFTER filtering:
    print((df_proper.count(), len(df_proper.columns)))

# Add the delay at previous stations

* We add now **two additional important features**.

* We take into consideration the delays at the **former 2 stations** of every station in a journey (when applicable).

* Objective: **df_ts** (Spark): contains all previous S-Bahn + weather information, and the delays at stations -1, -2.
* This is done in the background through **reshapeDf** (defined in the handlers).

In [44]:
# For details on reshapeDf(): Look in the handlers Noteboook for DataHandler.reshapeDf():
# - Reorders data according to SERVICE_ID (journey identifier)
#   and then according to the chronological order of events
# - Appends the columns prev0_VERSPAETUNG and prev1_prev0_VERSPAETUNG, that is,
#   delay at 1 or 2 previous stations (argument of the reshapeDf)
df_ts = DataHandler.reshapeDf(df_proper, 2)

In [45]:
# Commented out since Spark IO might cause kernel crash.
if (False):
    # Have a look at the 3 delay columns
    start_time = time.time()
    df_ts.select('VERSPAETUNG','prev0_VERSPAETUNG','prev1_prev0_VERSPAETUNG').show(n=10)
    print("--- %s seconds ---" % (time.time() - start_time))

# Create the df_ts_classification DataFrame

* Objective: df_ts_classification (Spark) = df_ts (created above) + binary column **{0,1}** to identify events **on time vs. delayed**

<a href='#Index'>Exercise Index</a>
<a id='Exercise-10'></a>

**EXERCISE 10**

* Add a binary column "classification" {0,1} according to delay {no,yes} at each station.

In [46]:
# Define a THRESHOLD for the delay in minutes:
threshold = 0

# Combine "when"+condition+"otherwise" to obtain a classification column {0,1}={on time,delayed} .
# The condition is VERSPAETUNG > resp. <= than the defined threshold:
df_ts_classification = df_ts.withColumn('classification', when(df_ts.VERSPAETUNG > threshold, 1).otherwise(0))

* Add the classification column {0,1} also for the delay at stations -1, -2.

In [47]:
# Do the same as above with prev0_VERSPAETUNG and prev1_prev0_VERSPAETUNG:
df_ts_classification = df_ts_classification.withColumn('prev0_classification', \
                      when(df_ts_classification.prev0_VERSPAETUNG > threshold, 1).otherwise(0))
df_ts_classification = df_ts_classification.withColumn('prev1_prev0_classification', \
                      when(df_ts_classification.prev1_prev0_VERSPAETUNG > threshold, 1).otherwise(0))

In [48]:
# Commented out since Spark IO might cause kernel crash.
if (False):
    # Have a look at the 3 columns for delay classification
    start_time = time.time()
    df_ts_classification.select('classification','prev0_classification','prev1_prev0_classification').show(n=10)
    print("--- %s seconds ---" % (time.time() - start_time))

# Create DFs with test and training subsets for ML algorithms

<a href='#Index'>Exercise Index</a>
<a id='Exercise-11'></a>

**EXERCISE 11**
* Randomly **split** the DF df_ts and df_ts_classification into **training** and **test** DataFrames.

* Objective: **df_train**, **df_test**, **df_train\_classification**, **df_test_classification** (Spark).


In [49]:
# Use randomSplit on both df_ts and df_ts_classification with weights and seed defined in the handlers.
(df_train, df_test) = df_ts.randomSplit(dfWeights[0], dfSeed_list[dfSeed_index])
(df_train_classification, df_test_classification) = df_ts_classification.randomSplit(dfWeights[0], dfSeed_list[dfSeed_index])

In [50]:
# Work in progress or in the script: Perform further splittings for the learning curve.

In [51]:
# Commented out since Spark IO might cause kernel crash.
if (False):
    # Check size of old and new dataframes:
    print((df_ts.count(), len(df_ts.columns)))
    print((df_train.count(), len(df_train.columns)))
    print((df_test.count(), len(df_test.columns)))

# Write everything down

* Generate a list with the directory path and the filename for each DataFrame that must be saved:

In [52]:
df_names= ['df_train'+suffix_s_w+'.csv', 'df_test'+suffix_s_w+'.csv', 'df_train_classification'+suffix_s_w+'.csv',
           'df_test_classification'+suffix_s_w+'.csv']

path_lustre = []
for filename in df_names:
    path_lustre.append(os.path.join(local_data_dir, filename))

## Long execution time!

* Writing down takes some time or might even crash with insufficient resources.
* All necessary DataFrames for the next Notebooks can be read-in from the folder NB_Dataframes_read_only and do not need to be overwritten.

In [53]:
if(False):
    print("Writing down all dfs from manipulation in... 1/", len(path_lustre))
    start_time = time.time()
    df_train.write.mode('overwrite').csv(os.path.join(LustrePrefix,path_lustre[0]), header = True)
    print("--- %.0f min. %.2f sec.---" % (np.floor_divide(time.time() - start_time, 60),
                                              np.remainder(time.time() - start_time, 60)))

In [54]:
if(False):
    print("Writing down all dfs from manipulation in... 2/", len(path_lustre))
    start_time = time.time()
    df_test.                write.mode('overwrite').csv(os.path.join(LustrePrefix,path_lustre[1]), header = True)
    print("--- %.0f min. %.2f sec.---" % (np.floor_divide(time.time() - start_time, 60),
                                              np.remainder(time.time() - start_time, 60)))

In [55]:
if(False):
    print("Writing down all dfs from manipulation in... 3/", len(path_lustre))
    start_time = time.time()
    df_train_classification.write.mode('overwrite').csv(os.path.join(LustrePrefix,path_lustre[2]), header = True)
    print("--- %.0f min. %.2f sec.---" % (np.floor_divide(time.time() - start_time, 60),
                                              np.remainder(time.time() - start_time, 60)))

In [56]:
if(False):
    print("Writing down all dfs from manipulation in... 4/", len(path_lustre))
    start_time = time.time()
    df_test_classification. write.mode('overwrite').csv(os.path.join(LustrePrefix,path_lustre[3]), header = True)
    print("--- %.0f min. %.2f sec.---" % (np.floor_divide(time.time() - start_time, 60),
                                              np.remainder(time.time() - start_time, 60)))